### Volgorde van de pipeline 


**Belangrijke termen**

**YAML:**  
Configuratie bestand waar alle cli_flags en default waardes en volgorde staan van de processors die gerunt moeten worden.  

**CLI:**  
Command Line Interface, De opties die de user in de terminal mee kan geven.

**CLI-flags:**  
De mogelijke opties zoals, -inpath, -ss, -elastic

**Arguments/args:**  
De echte input die een processor krijgt, bijvoorbeeld:  

args:  
  path:  
    cli: inpath  

De processor krijgt het argument "path" mee met de waarde de komt uit de CLI-optie "inpath" zoals test.pdb

**Defaults:**  
als de user niks meegeeft gebruikt argparse de waarde uit yaml, bijvoorbeeld:  
default: "martini3001"

**Processor:**  
Een stap binnen de pipeline die logica om het systeem uitvoerd en weer het bewerkte systeem teruggeeft, zoals:  
MakeBonds, DoMapping.

**System:**  
Het centrale object waar alles in zit: moleculen, atomen/beads, forcefield-info, metadata enzovoort. De hele pipeline verandert steeds ditzelfde system.

**Molecule:**  
Een molecuul binnen het system. In Vermouth is dit een graaf: nodes zijn atomen of beads, edges zijn verbindingen/bonds.

**Forcefield:**  
Set regels/parameters die bepalen hoe deeltjes in de simulatie zich gedragen. Het forcefield bepaalt ook welke pipeline-stappen nodig zijn.

**Mapping:**  
De vertaaltabel van atomistisch naar coarse-grained. Bijvoorbeeld: meerdere atomen worden samen één Martini bead.







### Pipeline

**Stap A:**  
Yaml wordt ingelezen. 

**Stap B:**  
Build_cli, maak de command line interface.  
Kijk in YAML welke cli-opties bestaan en maak daar argparse-argumenten van. 
Dus als je dit als yaml hebt:  
inpath:  
  type: path  
  required: true  

Dan zegt build_cli:  
parser.add_argument("-inpath", type = Path, required = True)  
Daarna snap het programma dat de gebruiker dit mag typen:  
-inpath test.pdb  

**Stap C:** 
parse_args() leest wat de gebruiker heeft getyped.    
parse_args() kijkt naar de terminal input.  
dus dit:  
python martiny.py -inpath test.pdb -bonds_from both  

wordt:
args.inpath = Path("test.pdb")  
args.bonds_from = "both"  

Daarna doe je:  
args = vars(args)
Je wilt er een dictionary van maken.  
Hier worden de defaults ingevuld, dus als de gebruiker bijvoorbeeld geen -to_ff heeft ingevuld dan wordt de default in de YAML gebruikt. Dus "to_ff": martini3001.  

**Stap D:**  
forcefields en mappings laden:   
known_force_fields  
known_mappings  
target_ff  
mappings  
Daarna stop je ze via variable_options in args, zodat yaml ze kan gebruiken als variables.  

**Stap E:**  
set_values_from_cli vult de yaml concreet in.  
voor set_values_cli is de yaml nog zo:  
args:  
  path:  
    cli: inpath  

Hierna wordt het zo:  
args:  
  path: Path("test.pdb")  
Ook rekent deze de conditions uit.  

**Stap F:**  
pipeline.from_json_conf maakt echte processor objecten aan, aan de hand van de args en kwargs.  
obj = processor(**kwargs) --> ReadSystem(path=Path("test.pdb"))  
Ook gebeurt hier bij mixins de vertaalslag. Dus de vertaalslag gebeurt tijdens het bouwen van het object en niet pas tijdens run_system.  










build_cli → bepaalt type van waardes  
parse_args → maakt die waardes (Path, float, bool, etc.)  
set_values_from_cli → zet die waardes in de pipeline structuur  
mixin → vertaalt waardes naar wat processor echt wil  
Processor object maken --> kwargs = conf['args'] --> obj = processor(**kwargs)